# 04 — Integración determinística de cohorte clínica y parámetros ECG

Este notebook integra la cohorte clínica anonimizada y segmentada con los parámetros extraídos desde los reportes ECG en PDF.

La integración **solo se ejecuta mediante una tabla puente validada** (`crosswalk_paciente_ecg.xlsx`) que relacione `PACIENTE_ID` o `REGISTRO_ID` con `clave_matching` del dataset ECG.

No se permite integración por variables débiles como edad, sexo, año, mes o fecha aproximada, porque dichas variables no identifican de manera única a un paciente ni a un evento clínico.

## 1. Entradas y salidas

### Entradas requeridas

```text
Base_Ordenada_Subsets_TFM.xlsx
crosswalk_paciente_ecg.xlsx
ecg_dataset.xlsx
```

### Salidas generadas

```text
Dataset_Multimodal_Integrado_TFM.xlsx
Auditoria_Integracion_ECG_TFM.xlsx
Reporte_Integracion_ECG_TFM.txt
```

### Regla metodológica

```text
Base clínica -> PACIENTE_ID / REGISTRO_ID
Crosswalk    -> PACIENTE_ID / REGISTRO_ID + clave_matching
ECG dataset  -> clave_matching
```

In [1]:
# Instalación de dependencias requeridas
# Ejecutar esta celda si el entorno no tiene alguna dependencia instalada.

import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "xlsxwriter": "XlsxWriter",
}

for modulo, paquete in DEPENDENCIAS.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia faltante: {paquete}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
    else:
        print(f"Dependencia disponible: {modulo}")

Dependencia disponible: pandas
Dependencia disponible: openpyxl
Dependencia disponible: xlsxwriter


In [2]:
# Librerías

from pathlib import Path
from datetime import datetime
import re
import unicodedata

import numpy as np
import pandas as pd

In [ ]:
# Configuración de rutas

BASE_DIR = Path(".")

PATH_BASE_CLINICA = BASE_DIR / "Base_Ordenada_Subsets_TFM.xlsx"
PATH_CROSSWALK = BASE_DIR / "crosswalk_paciente_ecg.xlsx"
PATH_ECG = BASE_DIR / "ecg_dataset.xlsx"

OUT_DATASET = BASE_DIR / "Dataset_Multimodal_Integrado_TFM.xlsx"
OUT_AUDITORIA = BASE_DIR / "Auditoria_Integracion_ECG_TFM.xlsx"
OUT_REPORTE = BASE_DIR / "Reporte_Integracion_ECG_TFM.txt"

print("Base clínica:", PATH_BASE_CLINICA.resolve())
print("Crosswalk:", PATH_CROSSWALK.resolve())
print("ECG dataset:", PATH_ECG.resolve())

## 2. Funciones auxiliares

In [4]:
def normalize_colname(col):
    """Normaliza nombres de columnas sin destruir la legibilidad."""
    s = str(col).strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def normalize_name(name):
    """Normaliza nombres para reproducir la clave usada por el notebook 03."""
    if pd.isna(name):
        return None
    s = str(name).strip()
    s = re.sub(r"\.pdf$", "", s, flags=re.I)
    s = re.sub(r"[\s_\-]+(\d+)\s*$", "", s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s).strip().upper()
    return s or None


def parse_fecha_segura(value):
    """Parsea fechas ISO y fechas latinas evitando advertencias por dayfirst."""
    if pd.isna(value):
        return pd.NaT
    s = str(value).strip()
    if not s:
        return pd.NaT
    if re.match(r"^\d{4}-\d{2}-\d{2}", s):
        return pd.to_datetime(s, errors="coerce", dayfirst=False)
    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def fecha_iso(value):
    d = parse_fecha_segura(value)
    if pd.isna(d):
        return None
    return d.strftime("%Y-%m-%d")


def require_file(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"No se encontró el archivo requerido: {path}. "
            "Ejecute primero los notebooks previos o copie el archivo a la carpeta de trabajo."
        )


def require_columns(df, required, df_name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} no contiene columnas requeridas: {missing}")


def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def add_registro_id_if_missing(df, prefix="REG"):
    if "REGISTRO_ID" not in df.columns:
        df = df.copy()
        df.insert(0, "REGISTRO_ID", [f"{prefix}_{i:08d}" for i in range(1, len(df) + 1)])
    return df


def normalize_bateria_columns(df):
    """Estandariza la trazabilidad de subconjuntos estructurales.

    SUBSET_EXAMENES se conserva como denominación original del proceso de segmentación.
    SUBSET_BATERIA se utiliza como alias estándar transversal para integración,
    construcción de endpoint, modelado y evaluación incremental.
    """
    if "SUBSET_BATERIA" not in df.columns and "SUBSET_EXAMENES" in df.columns:
        df["SUBSET_BATERIA"] = df["SUBSET_EXAMENES"]
    if "SUBSET_EXAMENES" not in df.columns and "SUBSET_BATERIA" in df.columns:
        df["SUBSET_EXAMENES"] = df["SUBSET_BATERIA"]
    return df


## 3. Verificación de archivos requeridos

El notebook detiene la ejecución si no existe `crosswalk_paciente_ecg.xlsx`, porque sin ese archivo no es posible asegurar la correspondencia paciente-a-paciente.

In [7]:
for path in [PATH_BASE_CLINICA, PATH_CROSSWALK, PATH_ECG]:
    require_file(path)

print("Archivos requeridos disponibles.")

Archivos requeridos disponibles.


## 4. Carga de base clínica segmentada

In [8]:
base_sheets = pd.read_excel(PATH_BASE_CLINICA, sheet_name=None)
print("Hojas detectadas en base clínica:", list(base_sheets.keys()))

if "BASE_COMPLETA" not in base_sheets:
    raise ValueError("Base_Ordenada_Subsets_TFM.xlsx debe contener una hoja BASE_COMPLETA.")

base_clinica = base_sheets["BASE_COMPLETA"].copy()
base_clinica.columns = [normalize_colname(c) for c in base_clinica.columns]
base_clinica = normalize_bateria_columns(base_clinica)
base_clinica = add_registro_id_if_missing(base_clinica, prefix="CLIN")

require_columns(base_clinica, ["PACIENTE_ID"], "BASE_COMPLETA")

if base_clinica["REGISTRO_ID"].duplicated().any():
    raise ValueError("REGISTRO_ID debe ser único en BASE_COMPLETA.")

print("Registros clínicos:", len(base_clinica))
print("Pacientes únicos:", base_clinica["PACIENTE_ID"].nunique(dropna=True))
base_clinica.head()

Hojas detectadas en base clínica: ['BASE_COMPLETA', 'BATERIA_A', 'BATERIA_B', 'BATERIA_C', 'BATERIA_D', 'RESUMEN_BATERIAS', 'METADATA', 'COLUMNAS_USADAS']
Registros clínicos: 3779
Pacientes únicos: 3779


,REGISTRO_ID,PACIENTE_ID,UltimaAtencion_Anio,UltimaAtencion_Mes,DiasDesdePrimeraAtencion,Edad,Sexo,Peso,Altura,IMC,...,FLAG_LDL_ALTO,FLAG_TRIGLICERIDOS_ALTOS,FLAG_OBESIDAD_IMC,FLAG_HDL_BAJO,TOTAL_RESULTADOS,TOTAL_VARIABLES_EVALUADAS,PORCENTAJE_COMPLETITUD,BATERIA_CLUSTER,SUBSET_EXAMENES,SUBSET_BATERIA
0,REG_000002,PAC_000002,2026.0,2.0,1227.0,48,Masculino,79 Kg,1.72 mts,26.70,...,1,1,0,0,31,31,100.0,2,BATERIA_A,BATERIA_A
1,REG_000007,PAC_000007,2025.0,2.0,847.0,59,Masculino,95 Kg,1.74 mts,31.38,...,1,1,1,0,31,31,100.0,2,BATERIA_A,BATERIA_A
2,REG_000008,PAC_000008,2025.0,4.0,910.0,28,Masculino,81 Kg,1.71 mts,27.70,...,1,1,0,0,31,31,100.0,2,BATERIA_A,BATERIA_A
3,REG_000015,PAC_000015,2025.0,8.0,1030.0,49,Masculino,98 Kg,1.70 mts,33.91,...,0,1,1,0,31,31,100.0,2,BATERIA_A,BATERIA_A
4,REG_000023,PAC_000023,2026.0,3.0,1234.0,38,Masculino,80 Kg,1.83 mts,24.38,...,0,0,0,0,31,31,100.0,2,BATERIA_A,BATERIA_A


## 5. Carga del crosswalk privado

El crosswalk es el artefacto privado que permite conectar el identificador anónimo de la cohorte clínica con la clave generada desde los ECG PDF.

In [9]:
crosswalk = pd.read_excel(PATH_CROSSWALK)
crosswalk.columns = [normalize_colname(c) for c in crosswalk.columns]

# Normalización defensiva de nombres esperados.
rename_map = {}
for col in crosswalk.columns:
    low = col.lower()
    if low == "paciente_id":
        rename_map[col] = "PACIENTE_ID"
    elif low == "registro_id":
        rename_map[col] = "REGISTRO_ID"
    elif low == "clave_matching":
        rename_map[col] = "clave_matching"
    elif low == "nombre_paciente_norm":
        rename_map[col] = "nombre_paciente_norm"
    elif low in {"fecha_atencion_norm", "fecha", "fecha_norm"}:
        rename_map[col] = "fecha_atencion_norm"

crosswalk = crosswalk.rename(columns=rename_map)

require_columns(crosswalk, ["PACIENTE_ID", "clave_matching"], "crosswalk_paciente_ecg")

if "REGISTRO_ID" in crosswalk.columns:
    key_clinica = "REGISTRO_ID"
else:
    key_clinica = "PACIENTE_ID"

crosswalk["clave_matching"] = crosswalk["clave_matching"].astype(str).str.strip()
crosswalk.loc[crosswalk["clave_matching"].isin(["", "nan", "None"]), "clave_matching"] = np.nan

print("Registros crosswalk:", len(crosswalk))
print("Clave clínica para integración:", key_clinica)
print("Claves ECG no nulas:", crosswalk["clave_matching"].notna().sum())
crosswalk.head()

Registros crosswalk: 3779
Clave clínica para integración: REGISTRO_ID
Claves ECG no nulas: 3773


,REGISTRO_ID,PACIENTE_ID,nombre_paciente_norm,fecha_atencion_norm,clave_matching,clave_hash_privada,estado_validacion,metodo_match,observaciones
0,REG_000001,PAC_000001,ALFARO ALFARO CAMILO NICOLAS,2025-07-17,ALFARO ALFARO CAMILO NICOLAS | 2025-07-17,5e388fa453ca93c575b13b13222e1963483a6524005276...,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN
1,REG_000002,PAC_000002,BARRERA MANCILLA RICHARD ALLAN,2026-02-26,BARRERA MANCILLA RICHARD ALLAN | 2026-02-26,a58003ea45242821f8fdb78c4d9d60629a0ba43a3d37a7...,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN
2,REG_000003,PAC_000003,CARLOS ENRIQUE VALCK TORO,2023-12-14,CARLOS ENRIQUE VALCK TORO | 2023-12-14,2da28545f8eecf27d3039d5742075aa58f17bdc42146b6...,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN
3,REG_000004,PAC_000004,CLAUDIO ALEJANDRO ESTAY CARVAJAL,2024-05-20,CLAUDIO ALEJANDRO ESTAY CARVAJAL | 2024-05-20,17b4d7906bb259aab41acb1ce2e4c862d861cf76158ab5...,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN
4,REG_000005,PAC_000005,CRISTIAN LEONARDO JUAREZ CORTES,2024-03-28,CRISTIAN LEONARDO JUAREZ CORTES | 2024-03-28,2f422a024707cb5d8e77a6b789b09551c59e9bf2782253...,pendiente_validacion,nombre_normalizado_fecha_atencion,NaN


## 6. Carga del dataset ECG

In [10]:
ecg = pd.read_excel(PATH_ECG)
ecg.columns = [normalize_colname(c) for c in ecg.columns]

# Normalización defensiva del dataset ECG.
if "ECG_ID" not in ecg.columns:
    ecg.insert(0, "ECG_ID", [f"ECG_{i:08d}" for i in range(1, len(ecg) + 1)])

if "clave_matching" not in ecg.columns:
    if {"nombre_paciente_norm", "fecha"}.issubset(ecg.columns):
        ecg["clave_matching"] = ecg["nombre_paciente_norm"].astype(str).str.strip() + " | " + ecg["fecha"].astype(str).str.strip()
    elif {"archivo_origen", "fecha"}.issubset(ecg.columns):
        ecg["nombre_paciente_norm"] = ecg["archivo_origen"].apply(normalize_name)
        ecg["clave_matching"] = ecg["nombre_paciente_norm"].astype(str).str.strip() + " | " + ecg["fecha"].astype(str).str.strip()
    else:
        raise ValueError("ecg_dataset.xlsx debe contener clave_matching o columnas para reconstruirla.")

if "fecha" not in ecg.columns and "fecha_examen" in ecg.columns:
    ecg["fecha"] = ecg["fecha_examen"].apply(fecha_iso)

if "rank_ecg_mismo_paciente_fecha" not in ecg.columns:
    ecg = ecg.sort_values(["clave_matching", "ECG_ID"]).copy()
    ecg["rank_ecg_mismo_paciente_fecha"] = ecg.groupby("clave_matching", dropna=False).cumcount() + 1

if "num_ecg_mismo_paciente_fecha" not in ecg.columns:
    ecg["num_ecg_mismo_paciente_fecha"] = ecg.groupby("clave_matching")["clave_matching"].transform("count").fillna(0).astype(int)

if "duplicado_mismo_dia" not in ecg.columns:
    ecg["duplicado_mismo_dia"] = ((ecg["num_ecg_mismo_paciente_fecha"] > 1) & ecg["clave_matching"].notna()).astype(int)

require_columns(ecg, ["ECG_ID", "clave_matching"], "ecg_dataset")

print("Registros ECG:", len(ecg))
print("Claves ECG no nulas:", ecg["clave_matching"].notna().sum())
print("Claves ECG únicas:", ecg["clave_matching"].nunique(dropna=True))
ecg.head()

Registros ECG: 2679
Claves ECG no nulas: 2678
Claves ECG únicas: 2623


,ECG_ID,archivo_origen,ruta_relativa,ano,mes,fecha_examen,paciente_id_pdf,sexo,edad,ECG_HR,...,observaciones_extraccion,nombre_paciente,nombre_paciente_norm,fecha,clave_matching,clave_hash_ecg,num_ecg_mismo_paciente_fecha,duplicado_mismo_dia,rank_ecg_mismo_paciente_fecha,estado_match_ecg
0,ECG_000001,BRENDA OCHOA MARTINEZ 2.pdf,2024/ABRIL/01/BRENDA OCHOA MARTINEZ 2.pdf,2024,4.0,01-04-2024 9:31:50,3,M,37.0,59.0,...,NaN,BRENDA OCHOA MARTINEZ 2,BRENDA OCHOA MARTINEZ,2024-04-01,BRENDA OCHOA MARTINEZ | 2024-04-01,a1c2680594f62f63968ac3ec3212bcea45a17d8bc055bf...,2,1,1,CLAVE_OK_PARAMETROS_CORE_OK
1,ECG_000002,BRENDA OCHOA MARTINEZ.pdf,2024/ABRIL/01/BRENDA OCHOA MARTINEZ.pdf,2024,4.0,01-04-2024 9:28:45,3,M,37.0,63.0,...,NaN,BRENDA OCHOA MARTINEZ,BRENDA OCHOA MARTINEZ,2024-04-01,BRENDA OCHOA MARTINEZ | 2024-04-01,a1c2680594f62f63968ac3ec3212bcea45a17d8bc055bf...,2,1,2,CLAVE_OK_PARAMETROS_CORE_OK
2,ECG_000003,FELIPE AQUEZ PERALTA.pdf,2024/ABRIL/01/FELIPE AQUEZ PERALTA.pdf,2024,4.0,01-04-2024 9:52:34,3,M,35.0,73.0,...,NaN,FELIPE AQUEZ PERALTA,FELIPE AQUEZ PERALTA,2024-04-01,FELIPE AQUEZ PERALTA | 2024-04-01,4d2e337db087e62fd28e7974260ba7482962178e8532c3...,1,0,1,CLAVE_OK_PARAMETROS_CORE_OK
3,ECG_000004,JAIME GUZMAN COLLAO.pdf,2024/ABRIL/01/JAIME GUZMAN COLLAO.pdf,2024,4.0,01-04-2024 11:23:09,3,M,37.0,55.0,...,NaN,JAIME GUZMAN COLLAO,JAIME GUZMAN COLLAO,2024-04-01,JAIME GUZMAN COLLAO | 2024-04-01,e3923bfb9994c56ad5a1ecd142c1f59992440f1d3548e7...,1,0,1,CLAVE_OK_PARAMETROS_CORE_OK
4,ECG_000005,CRISTIAN GUZMAN ASENCIO.pdf,2024/ABRIL/02/CRISTIAN GUZMAN ASENCIO.pdf,2024,4.0,02-04-2024 8:27:40,3,M,54.0,46.0,...,NaN,CRISTIAN GUZMAN ASENCIO,CRISTIAN GUZMAN ASENCIO,2024-04-02,CRISTIAN GUZMAN ASENCIO | 2024-04-02,56711961622f80a4bbe4d6942ca9c4e532991cd4cb74e7...,1,0,1,CLAVE_OK_PARAMETROS_CORE_OK


## 7. Validaciones de unicidad y cobertura

In [11]:
validaciones = []

def add_validation(nombre, valor, estado, detalle=""):
    validaciones.append({
        "validacion": nombre,
        "valor": valor,
        "estado": estado,
        "detalle": detalle,
    })

add_validation("base_registros", len(base_clinica), "OK")
add_validation("base_pacientes_unicos", base_clinica["PACIENTE_ID"].nunique(dropna=True), "OK")
add_validation("crosswalk_registros", len(crosswalk), "OK")
add_validation("crosswalk_clave_matching_no_nula", int(crosswalk["clave_matching"].notna().sum()), "OK")
add_validation("ecg_registros", len(ecg), "OK")
add_validation("ecg_clave_matching_no_nula", int(ecg["clave_matching"].notna().sum()), "OK")

# Duplicados potenciales en crosswalk.
if "REGISTRO_ID" in crosswalk.columns:
    dup_reg = int(crosswalk["REGISTRO_ID"].duplicated().sum())
    add_validation("crosswalk_REGISTRO_ID_duplicados", dup_reg, "OK" if dup_reg == 0 else "REVISAR")

crosswalk_dup_clave = int(crosswalk["clave_matching"].duplicated().sum())
add_validation(
    "crosswalk_clave_matching_duplicados",
    crosswalk_dup_clave,
    "OK" if crosswalk_dup_clave == 0 else "REVISAR",
    "Puede ser válido si existe más de un evento clínico enlazado al mismo ECG; debe revisarse."
)

ecg_dup_clave = int(ecg["clave_matching"].duplicated().sum())
add_validation(
    "ecg_clave_matching_duplicados",
    ecg_dup_clave,
    "OK" if ecg_dup_clave == 0 else "REVISAR",
    "Duplicados ECG mismo paciente-fecha se resuelven usando rank_ecg_mismo_paciente_fecha == 1 para dataset principal."
)

validaciones_df = pd.DataFrame(validaciones)
validaciones_df

,validacion,valor,estado,detalle
0,base_registros,3779,OK,
1,base_pacientes_unicos,3779,OK,
2,crosswalk_registros,3779,OK,
3,crosswalk_clave_matching_no_nula,3773,OK,
4,ecg_registros,2679,OK,
5,ecg_clave_matching_no_nula,2678,OK,
6,crosswalk_REGISTRO_ID_duplicados,0,OK,
7,crosswalk_clave_matching_duplicados,5,REVISAR,Puede ser válido si existe más de un evento cl...
8,ecg_clave_matching_duplicados,55,REVISAR,Duplicados ECG mismo paciente-fecha se resuelv...


## 8. Resolución controlada de ECG duplicados

Para el dataset principal se conserva un único ECG por `clave_matching`.

Si existen varios ECG para el mismo paciente y fecha, se conserva el registro con `rank_ecg_mismo_paciente_fecha == 1` y se exportan los duplicados a auditoría.

In [12]:
ecg_principal = ecg[ecg["clave_matching"].notna()].copy()
ecg_principal = ecg_principal.sort_values(["clave_matching", "rank_ecg_mismo_paciente_fecha", "ECG_ID"])

ecg_duplicados_revision = ecg_principal[ecg_principal.duplicated("clave_matching", keep=False)].copy()
ecg_principal = ecg_principal.drop_duplicates("clave_matching", keep="first").copy()

print("ECG disponibles para integración principal:", len(ecg_principal))
print("ECG enviados a revisión por duplicidad:", len(ecg_duplicados_revision))

ECG disponibles para integración principal: 2623
ECG enviados a revisión por duplicidad: 108


## 9. Integración determinística

In [13]:
# 1) Base clínica + crosswalk
if key_clinica == "REGISTRO_ID":
    require_columns(base_clinica, ["REGISTRO_ID"], "BASE_COMPLETA")
    clinica_crosswalk = base_clinica.merge(
        crosswalk,
        on="REGISTRO_ID",
        how="left",
        suffixes=("", "_crosswalk"),
        indicator="_merge_clinica_crosswalk",
    )
else:
    clinica_crosswalk = base_clinica.merge(
        crosswalk,
        on="PACIENTE_ID",
        how="left",
        suffixes=("", "_crosswalk"),
        indicator="_merge_clinica_crosswalk",
    )

# 2) Resultado + ECG por clave_matching
integrado = clinica_crosswalk.merge(
    ecg_principal,
    on="clave_matching",
    how="left",
    suffixes=("", "_ecg"),
    indicator="_merge_crosswalk_ecg",
)

integrado = normalize_bateria_columns(integrado)

integrado["flag_crosswalk_disponible"] = integrado["clave_matching"].notna().astype(int)
integrado["flag_ecg_integrado"] = integrado["ECG_ID"].notna().astype(int)

print("Registros integrados:", len(integrado))
print("Con crosswalk:", int(integrado["flag_crosswalk_disponible"].sum()))
print("Con ECG integrado:", int(integrado["flag_ecg_integrado"].sum()))
integrado.head()

Registros integrados: 3779
Con crosswalk: 3773
Con ECG integrado: 48


,REGISTRO_ID,PACIENTE_ID,UltimaAtencion_Anio,UltimaAtencion_Mes,DiasDesdePrimeraAtencion,Edad,Sexo,Peso,Altura,IMC,...,nombre_paciente_norm_ecg,fecha,clave_hash_ecg,num_ecg_mismo_paciente_fecha,duplicado_mismo_dia,rank_ecg_mismo_paciente_fecha,estado_match_ecg,_merge_crosswalk_ecg,flag_crosswalk_disponible,flag_ecg_integrado
0,REG_000002,PAC_000002,2026.0,2.0,1227.0,48,Masculino,79 Kg,1.72 mts,26.70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1,0
1,REG_000007,PAC_000007,2025.0,2.0,847.0,59,Masculino,95 Kg,1.74 mts,31.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1,0
2,REG_000008,PAC_000008,2025.0,4.0,910.0,28,Masculino,81 Kg,1.71 mts,27.70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1,0
3,REG_000015,PAC_000015,2025.0,8.0,1030.0,49,Masculino,98 Kg,1.70 mts,33.91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1,0
4,REG_000023,PAC_000023,2026.0,3.0,1234.0,38,Masculino,80 Kg,1.83 mts,24.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1,0


## 10. Construcción del dataset limpio para modelado

Se eliminan columnas sensibles o potencialmente reidentificables del dataset principal. El detalle de trazabilidad se conserva solo en el archivo de auditoría privada.

In [14]:
cols_sensibles_patrones = [
    "nombre",
    "apellido",
    "rut",
    "documento",
    "telefono",
    "email",
    "direccion",
    "clave_matching",
    "clave_hash",
    "archivo_origen",
    "ruta_relativa",
    "nombre_paciente",
]

cols_drop = []
for c in integrado.columns:
    low = c.lower()
    if any(p in low for p in cols_sensibles_patrones):
        cols_drop.append(c)

# Mantener variables técnicas no reidentificables y variables ECG.
dataset_modelado = integrado.drop(columns=sorted(set(cols_drop)), errors="ignore").copy()
dataset_modelado = normalize_bateria_columns(dataset_modelado)

# Evitar columnas de merge internas en dataset de modelado.
dataset_modelado = dataset_modelado.drop(
    columns=[c for c in dataset_modelado.columns if c.startswith("_merge_")],
    errors="ignore"
)

print("Columnas eliminadas por privacidad:", sorted(set(cols_drop)))
print("Dimensión dataset modelado:", dataset_modelado.shape)
dataset_modelado.head()

Columnas eliminadas por privacidad: ['archivo_origen', 'clave_hash_ecg', 'clave_hash_privada', 'clave_matching', 'nombre_paciente', 'nombre_paciente_norm', 'nombre_paciente_norm_ecg', 'ruta_relativa']
Dimensión dataset modelado: (3779, 101)


,REGISTRO_ID,PACIENTE_ID,UltimaAtencion_Anio,UltimaAtencion_Mes,DiasDesdePrimeraAtencion,Edad,Sexo,Peso,Altura,IMC,...,pdf_valido,parametros_extraidos,observaciones_extraccion,fecha,num_ecg_mismo_paciente_fecha,duplicado_mismo_dia,rank_ecg_mismo_paciente_fecha,estado_match_ecg,flag_crosswalk_disponible,flag_ecg_integrado
0,REG_000002,PAC_000002,2026.0,2.0,1227.0,48,Masculino,79 Kg,1.72 mts,26.70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
1,REG_000007,PAC_000007,2025.0,2.0,847.0,59,Masculino,95 Kg,1.74 mts,31.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,REG_000008,PAC_000008,2025.0,4.0,910.0,28,Masculino,81 Kg,1.71 mts,27.70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
3,REG_000015,PAC_000015,2025.0,8.0,1030.0,49,Masculino,98 Kg,1.70 mts,33.91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
4,REG_000023,PAC_000023,2026.0,3.0,1234.0,38,Masculino,80 Kg,1.83 mts,24.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0


## 11. Integración por pseudo-baterías

Cada hoja `BATERIA_*` se filtra desde el dataset principal usando `REGISTRO_ID` cuando está disponible. Esto evita volver a ejecutar joins separados y mantiene consistencia con `BASE_COMPLETA`.

La trazabilidad estructural se conserva mediante `SUBSET_EXAMENES` y `SUBSET_BATERIA`, sin usar estas columnas como variables clínicas ni electrocardiográficas.

In [15]:
bateria_sheets_integradas = {}

for sheet_name, df_sheet in base_sheets.items():
    if sheet_name == "BASE_COMPLETA":
        continue
    if not sheet_name.upper().startswith("BATERIA"):
        continue

    tmp = df_sheet.copy()
    tmp.columns = [normalize_colname(c) for c in tmp.columns]
    tmp = normalize_bateria_columns(tmp)

    if "REGISTRO_ID" in tmp.columns and "REGISTRO_ID" in dataset_modelado.columns:
        subset = dataset_modelado[dataset_modelado["REGISTRO_ID"].isin(tmp["REGISTRO_ID"])].copy()
    elif "PACIENTE_ID" in tmp.columns and "PACIENTE_ID" in dataset_modelado.columns:
        subset = dataset_modelado[dataset_modelado["PACIENTE_ID"].isin(tmp["PACIENTE_ID"])].copy()
    else:
        subset = pd.DataFrame()

    if not subset.empty:
        subset = normalize_bateria_columns(subset)
        if "SUBSET_BATERIA" not in subset.columns:
            subset["SUBSET_BATERIA"] = sheet_name
        if "SUBSET_EXAMENES" not in subset.columns:
            subset["SUBSET_EXAMENES"] = subset["SUBSET_BATERIA"]

    bateria_sheets_integradas[sheet_name] = subset
    print(sheet_name, subset.shape)

BATERIA_A (936, 101)
BATERIA_B (1192, 101)
BATERIA_C (837, 101)
BATERIA_D (814, 101)


## 12. Indicadores de cobertura de integración

In [16]:
total_registros = len(integrado)
con_crosswalk = int(integrado["flag_crosswalk_disponible"].sum())
con_ecg = int(integrado["flag_ecg_integrado"].sum())

resumen_integracion = pd.DataFrame([
    {"indicador": "registros_clinicos", "valor": total_registros},
    {"indicador": "registros_con_crosswalk", "valor": con_crosswalk},
    {"indicador": "registros_con_ecg_integrado", "valor": con_ecg},
    {"indicador": "porcentaje_con_crosswalk", "valor": round(con_crosswalk / total_registros * 100, 2) if total_registros else 0},
    {"indicador": "porcentaje_con_ecg_integrado", "valor": round(con_ecg / total_registros * 100, 2) if total_registros else 0},
    {"indicador": "ecg_duplicados_revision", "valor": len(ecg_duplicados_revision)},
])

# Cobertura por batería si existe columna de batería.
bateria_col = first_existing_column(dataset_modelado, ["SUBSET_BATERIA", "SUBSET_EXAMENES", "BATERIA", "BATERIA_ESTRUCTURAL", "BATERIA_CLUSTER", "CLUSTER_BATERIA"])
if bateria_col:
    cobertura_bateria = (
        dataset_modelado
        .groupby(bateria_col, dropna=False)
        .agg(
            registros=("PACIENTE_ID", "size"),
            con_ecg=("flag_ecg_integrado", "sum")
        )
        .reset_index()
    )
    if bateria_col != "SUBSET_BATERIA":
        cobertura_bateria = cobertura_bateria.rename(columns={bateria_col: "SUBSET_BATERIA"})
    cobertura_bateria["porcentaje_con_ecg"] = (cobertura_bateria["con_ecg"] / cobertura_bateria["registros"] * 100).round(2)
else:
    cobertura_bateria = pd.DataFrame(columns=["bateria", "registros", "con_ecg", "porcentaje_con_ecg"])

resumen_integracion, cobertura_bateria

(                      indicador    valor
 0            registros_clinicos  3779.00
 1       registros_con_crosswalk  3773.00
 2   registros_con_ecg_integrado    48.00
 3      porcentaje_con_crosswalk    99.84
 4  porcentaje_con_ecg_integrado     1.27
 5       ecg_duplicados_revision   108.00,
   SUBSET_BATERIA  registros  con_ecg  porcentaje_con_ecg
 0      BATERIA_A        936       16                1.71
 1      BATERIA_B       1192       17                1.43
 2      BATERIA_C        837       15                1.79
 3      BATERIA_D        814        0                0.00)

## 13. Exportación de resultados

In [ ]:
with pd.ExcelWriter(OUT_DATASET, engine="xlsxwriter") as writer:
    dataset_modelado.to_excel(writer, sheet_name="BASE_MULTIMODAL", index=False)
    for sheet_name, df_sheet in bateria_sheets_integradas.items():
        safe_sheet = sheet_name[:31]
        df_sheet.to_excel(writer, sheet_name=safe_sheet, index=False)
    resumen_integracion.to_excel(writer, sheet_name="RESUMEN_INTEGRACION", index=False)
    cobertura_bateria.to_excel(writer, sheet_name="COBERTURA_BATERIA", index=False)

with pd.ExcelWriter(OUT_AUDITORIA, engine="xlsxwriter") as writer:
    validaciones_df.to_excel(writer, sheet_name="VALIDACIONES", index=False)
    resumen_integracion.to_excel(writer, sheet_name="RESUMEN", index=False)
    cobertura_bateria.to_excel(writer, sheet_name="COBERTURA_BATERIA", index=False)
    crosswalk.to_excel(writer, sheet_name="CROSSWALK_USADO", index=False)
    ecg_duplicados_revision.to_excel(writer, sheet_name="ECG_DUPLICADOS", index=False)
    integrado.to_excel(writer, sheet_name="INTEGRACION_AUDITORIA", index=False)

print("Archivo dataset:", OUT_DATASET.resolve())
print("Archivo auditoría:", OUT_AUDITORIA.resolve())

## 14. Reporte textual de integración

In [ ]:
lines = []
lines.append("REPORTE INTEGRACION ECG TFM")
lines.append("=" * 70)
lines.append(f"Fecha generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append("")
lines.append("ARCHIVOS DE ENTRADA")
lines.append("-" * 70)
lines.append(f"Base clínica : {PATH_BASE_CLINICA}")
lines.append(f"Crosswalk    : {PATH_CROSSWALK}")
lines.append(f"ECG dataset  : {PATH_ECG}")
lines.append("")
lines.append("ESTRATEGIA DE INTEGRACION")
lines.append("-" * 70)
lines.append("Estrategia: DETERMINISTICA_CON_CROSSWALK")
lines.append(f"Clave clínica usada: {key_clinica}")
lines.append("Clave ECG usada: clave_matching")
lines.append("Regla: no se integran registros por edad, sexo, año, mes ni fecha aproximada.")
lines.append("")
lines.append("RESUMEN")
lines.append("-" * 70)
for _, r in resumen_integracion.iterrows():
    lines.append(f"{r['indicador']}: {r['valor']}")
lines.append("")
lines.append("VALIDACIONES")
lines.append("-" * 70)
for _, r in validaciones_df.iterrows():
    lines.append(f"{r['validacion']}: {r['valor']} [{r['estado']}] {r.get('detalle', '')}")
lines.append("")
lines.append("SALIDAS")
lines.append("-" * 70)
lines.append(f"Dataset multimodal: {OUT_DATASET}")
lines.append(f"Auditoría privada : {OUT_AUDITORIA}")
lines.append(f"Reporte TXT       : {OUT_REPORTE}")
lines.append("")
lines.append("NOTA DE PRIVACIDAD")
lines.append("-" * 70)
lines.append("El dataset principal elimina columnas potencialmente reidentificables.")
lines.append("El archivo de auditoría puede contener claves sensibles o derivadas de nombres y no debe publicarse.")

OUT_REPORTE.write_text("\n".join(lines), encoding="utf-8")
print("Reporte generado:", OUT_REPORTE.resolve())
print("\n".join(lines[:30]))

## 15. Estado final del proceso

El proceso produce un dataset multimodal válido para etapas posteriores solo si existe cobertura ECG integrada mediante crosswalk validado.

Si `registros_con_ecg_integrado` es 0, el archivo resultante debe interpretarse como salida técnica sin valor para modelado multimodal.

In [19]:
if con_ecg == 0:
    raise ValueError(
        "La integración finalizó sin registros ECG enlazados. "
        "No usar Dataset_Multimodal_Integrado_TFM.xlsx para modelado multimodal. "
        "Revise crosswalk_paciente_ecg.xlsx y ecg_dataset.xlsx."
    )

print("Integración finalizada con registros ECG enlazados.")

Integración finalizada con registros ECG enlazados.
